# MNIST demo with CREST 

The entire notebook should run in < 60 seconds, and use ~600MB of memory.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

#### Load MNIST

In [3]:
import tensorflow as tf
import numpy as np

# Load data and define constants
xy,xy_test = tf.keras.datasets.mnist.load_data() 
xy_train   = [i[:50000]for i in xy]
xy_valid   = [i[50000:]for i in xy]
n_classes  = 10
patch_size = 28

In [4]:
len(xy)

2

#### Create the datasets

In [5]:
from crest.data.loading.StructuredDataset import StructuredDataset

# Set input/output key labels
INP = 'image'
OUT = 'class'

# Define the train/test datasets, as well as the array labels
ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_valid = StructuredDataset(*xy_valid, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

#### Create the batchers

In [6]:
from crest.data.Batcher import Batcher

# Note: workers=0 is faster for this, but may cause issues in a notebook
# since the background threads will run in the main process 
batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}
batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)
batch_test  = Batcher(ds_test ,  **batch_kwargs)

In [7]:
a = next(batch_test)

#### Create the CREST graph

In [8]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : tf.TensorSpec(shape=[None,patch_size, patch_size])},
    outputs={OUT : tf.TensorSpec(shape=[None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

[None, None, None, None, None, None, None]


#### Build, train, and test using Model

In [9]:
from crest.model.Model import Model

build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : 'sparse_categorical_accuracy'
}

model = Model(graph)
model.build(**build_kwargs)
model.fit(batch_train, **{
    'validation_data'  : batch_valid,
    'steps_per_epoch'  : 250, # 50k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1,
})


Epoch 1/5
  8/250 [..............................] - ETA: 1s - loss: 21.5942 - sparse_categorical_accuracy: 0.4350 

2023-10-30 10:19:33.281453: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


250/250 [==============================] - 3s 13ms/step - loss: 2.3707 - sparse_categorical_accuracy: 0.8836 - val_loss: 0.3340 - val_sparse_categorical_accuracy: 0.9605
Epoch 2/5
250/250 [==============================] - 3s 13ms/step - loss: 0.3235 - sparse_categorical_accuracy: 0.9448 - val_loss: 0.1664 - val_sparse_categorical_accuracy: 0.9674
Epoch 3/5
250/250 [==============================] - 3s 13ms/step - loss: 0.1619 - sparse_categorical_accuracy: 0.9608 - val_loss: 0.1377 - val_sparse_categorical_accuracy: 0.9727
Epoch 4/5
250/250 [==============================] - 3s 13ms/step - loss: 0.1084 - sparse_categorical_accuracy: 0.9692 - val_loss: 0.1213 - val_sparse_categorical_accuracy: 0.9738
Epoch 5/5
250/250 [==============================] - 3s 12ms/step - loss: 0.0821 - sparse_categorical_accuracy: 0.9749 - val_loss: 0.1200 - val_sparse_categorical_accuracy: 0.9726


In [10]:
# Evaluate the model on the test set
accuracy = model.evaluate(batch_test,verbose=False,return_dict=True)
print(f'Test performance metrics: {accuracy}')

Test performance metrics: {'loss': 0.11389175057411194, 'sparse_categorical_accuracy': 0.9599999785423279}


Accuracy on the full test set should be > ~97%.

In [11]:
# Close the batchers
batch_train.close()
batch_valid.close()
batch_test.close()

### Make predictions

In [12]:
x_pred,_ = xy_test  # Use images from the testing set
ds_pred = StructuredDataset(*[x_pred], labels=[INP])
batch_pred = Batcher(ds_pred, **{
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
})

In [13]:
# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False
}

In [14]:
# Generate predictions on the whole test set
pred_kwargs['steps'] = 10000//pred_kwargs['batch_size']
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
    'shuffle'    : False
}) 
model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


Adding auxillary output like lat and lon when making a prediction

In [15]:
# Combine some fictous auxillary data with testing input for prediction
x,_ = xy_test
lat = np.arange(len(x),dtype=float)
lon = np.arange(len(x),dtype=float)
x_pred = (lat,lon,x)

In [16]:
# Create the prediction data and make predictions
ds_pred = StructuredDataset(*x_pred, labels=['lat','lon',INP])
aux_pred = model.predict(ds_pred,coords=['lat','lon'],**pred_kwargs)

In [17]:
# Check you have lat,lon now in your output
aux_pred 

{'class': array([[3.0616324e-13, 1.8469525e-17, 3.8184933e-12, ..., 9.9999356e-01,
         6.7440609e-11, 1.6785838e-09],
        [8.5880580e-11, 2.2145832e-07, 9.9999905e-01, ..., 2.4220791e-17,
         1.3221301e-08, 3.0367930e-20],
        [3.5355981e-08, 9.9992990e-01, 2.6993665e-05, ..., 4.7477749e-07,
         2.5115167e-05, 1.7040843e-06],
        ...,
        [4.3601925e-13, 3.7941345e-11, 4.2923134e-14, ..., 4.2989655e-07,
         1.0789679e-06, 1.3568964e-05],
        [1.4264012e-10, 8.5279785e-15, 2.5119377e-12, ..., 1.1081096e-11,
         1.3089596e-02, 1.8928119e-11],
        [5.0669380e-10, 8.0363830e-14, 1.2540170e-09, ..., 6.8261030e-14,
         7.3944011e-11, 1.8550566e-13]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}